# PHÂN TÍCH ĐÁNH GIÁ KHÁCH HÀNG — DEEP LEARNING (PYTORCH MLP)
**Bài toán:** Phân loại cảm xúc/khuyến nghị (Binary Classification)
---
**Mô hình:** Mạng Nơ-ron Phân loại Văn bản (Text Classification MLP with PyTorch).

In [ ]:
import numpy as np
import pandas as pd
import joblib, os, time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

MODEL_DIR = os.path.join('..', 'models')
data = np.load(os.path.join(MODEL_DIR, 'tfidf_data.npz'))
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

X_tr_t = torch.tensor(X_train, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.long)
X_va_t = torch.tensor(X_val, dtype=torch.float32)
y_va_t = torch.tensor(y_val, dtype=torch.long)
X_te_t = torch.tensor(X_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_va_t, y_va_t), batch_size=128, shuffle=False)


In [ ]:
class TextClassifierMLP(nn.Module):
    def __init__(self, in_features, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
model = TextClassifierMLP(X_train.shape[1], num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

for epoch in range(1, 16):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_va_t).argmax(dim=1).numpy()
        val_acc = accuracy_score(y_val, val_preds)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}: Train Loss = {total_loss/len(X_train):.4f} | Val Acc = {val_acc:.4f}')


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(X_te_t)
    probs = torch.softmax(logits, dim=1)[:, 1].numpy()
    preds = logits.argmax(dim=1).numpy()

acc = accuracy_score(y_test, preds)
prec = precision_score(y_test, preds, zero_division=0)
rec = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)
auc = roc_auc_score(y_test, probs)

print(f"Deep Learning (PyTorch MLP) -> Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'cb_mlp_best.pth'))
np.savez_compressed(os.path.join(MODEL_DIR, 'cb_dl_preds.npz'), preds=preds, probs=probs)
print('Saved PyTorch Model.')
